# 01 — Exploratory Data Analysis
r/soccer World Cup 2022 — EDA


In [ ]:
# Shared Setup — run this first
import json, pandas as pd, numpy as np
import matplotlib.pyplot as plt, matplotlib.dates as mdates
import seaborn as sns, warnings, re
from collections import Counter
from datetime import datetime
from tqdm import tqdm
warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["figure.dpi"] = 120
sns.set_style("whitegrid")

POSTS_FILE    = "../data/posts_soccer.json"
COMMENTS_FILE = "../data/comments_soccer.json"

with open(POSTS_FILE, "r", encoding="utf-8") as f:
    posts_raw = json.load(f)
with open(COMMENTS_FILE, "r", encoding="utf-8") as f:
    comments_raw = json.load(f)

posts_df = pd.DataFrame([{
    "id": p.get("id"), "author": p.get("author"),
    "title": p.get("title",""), "selftext": p.get("selftext",""),
    "score": p.get("score",0), "upvote_ratio": p.get("upvote_ratio",0),
    "num_comments": p.get("num_comments",0), "subreddit": p.get("subreddit"),
    "flair": p.get("link_flair_text"), "created_utc": p.get("created_utc"),
} for p in posts_raw])

comments_df = pd.DataFrame([{
    "id": c.get("id"), "author": c.get("author"),
    "body": c.get("body",""), "score": c.get("score",0),
    "parent_id": c.get("parent_id"), "link_id": c.get("link_id"),
    "subreddit": c.get("subreddit"), "created_utc": c.get("created_utc"),
    "controversiality": c.get("controversiality",0),
} for c in comments_raw])

REMOVE = [None, "[deleted]", "AutoModerator", "BotDefense"]
posts_df["datetime"]    = pd.to_datetime(posts_df["created_utc"], unit="s")
comments_df["datetime"] = pd.to_datetime(comments_df["created_utc"], unit="s")
posts_df    = posts_df[~posts_df["author"].isin(REMOVE)].reset_index(drop=True)
comments_df = comments_df[~comments_df["author"].isin(REMOVE)].reset_index(drop=True)
posts_df["full_text"] = posts_df["title"] + " " + posts_df["selftext"].fillna("")

print(f"Posts: {len(posts_df):,} | Comments: {len(comments_df):,}")
print(f"Date range: {posts_df["datetime"].min().date()} to {posts_df["datetime"].max().date()}")


## 2. Exploratory Data Analysis

In [ ]:
# ── POSTING ACTIVITY OVER TIME ─────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Posts per day
posts_per_day = posts_df.set_index('datetime').resample('D')['id'].count()
axes[0].plot(posts_per_day.index, posts_per_day.values, color='#e63946', linewidth=2)
axes[0].fill_between(posts_per_day.index, posts_per_day.values, alpha=0.2, color='#e63946')
axes[0].set_title('Daily Post Volume on r/soccer — World Cup 2022', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Number of Posts')
axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))

# Comments per day
comments_per_day = comments_df.set_index('datetime').resample('D')['id'].count()
axes[1].plot(comments_per_day.index, comments_per_day.values, color='#457b9d', linewidth=2)
axes[1].fill_between(comments_per_day.index, comments_per_day.values, alpha=0.2, color='#457b9d')
axes[1].set_title('Daily Comment Volume on r/soccer — World Cup 2022', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Number of Comments')
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))

plt.tight_layout()
plt.savefig('activity_over_time.png', bbox_inches='tight')
plt.show()
print('Saved: activity_over_time.png')

In [ ]:
# ── POST FLAIR DISTRIBUTION ────────────────────────────────────────────────
flair_counts = posts_df['flair'].value_counts().head(10)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Flair bar chart
flair_counts.plot(kind='barh', ax=axes[0], color='#e63946')
axes[0].set_title('Top 10 Post Flairs', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Number of Posts')
axes[0].invert_yaxis()

# Score distribution
axes[1].hist(posts_df['score'].clip(0, 500), bins=50, color='#457b9d', edgecolor='white')
axes[1].set_title('Post Score Distribution (clipped at 500)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Score (upvotes)')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.savefig('eda_distributions.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── TOP POSTS ──────────────────────────────────────────────────────────────
print('=== TOP 10 POSTS BY SCORE ===')
top_posts = posts_df.nlargest(10, 'score')[['title', 'score', 'num_comments', 'flair', 'datetime']]
top_posts['title'] = top_posts['title'].str[:80]
print(top_posts.to_string(index=False))

print('\n=== MOST ACTIVE POSTERS ===')
print(posts_df['author'].value_counts().head(10))